# Практична робота №2: Вступ до машинного навчання (Kaggle Tutorial)
Виконання етапів 2-7: побудова та валідація моделей для прогнозування вартості нерухомості.

### Етап 2: Завантаження та базовий аналіз даних (Basic Data Exploration)

In [1]:
import pandas as pd


melbourne_file_path = 'melb_data.csv'
melbourne_data = pd.read_csv(melbourne_file_path)

# Очищення від рядків з пропущеними значеннями
melbourne_data = melbourne_data.dropna(axis=0)

print("Розмірність датасету після очищення:", melbourne_data.shape)
melbourne_data.describe()

Розмірність датасету після очищення: (1000, 21)


,Rooms,Price,Distance,Postcode,Bedroom2,Bathroom,Car,Landsize,BuildingArea,YearBuilt,Lattitude,Longtitude,Propertycount
count,1000.000000,1.000000e+03,1000.00000,1000.0,1000.000000,1000.000000,1000.0,1000.000000,1000.000000,1000.0,1000.000000,1000.000000,1000.0
mean,3.004000,1.250340e+06,8.21400,3000.0,3.004000,1.986000,1.0,545.331000,245.402300,1985.0,-37.849398,144.997853,4019.0
std,1.433192,3.974116e+05,3.32196,0.0,1.433192,0.815149,0.0,237.458784,106.856571,0.0,0.086933,0.115145,0.0
min,1.000000,3.483000e+05,2.50000,3000.0,1.000000,1.000000,1.0,120.000000,54.000000,1985.0,-38.000000,144.800260,4019.0
25%,2.000000,9.271500e+05,5.30000,3000.0,2.000000,1.000000,1.0,339.000000,152.600000,1985.0,-37.918655,144.896600,4019.0
50%,3.000000,1.240000e+06,8.20000,3000.0,3.000000,2.000000,1.0,555.000000,249.800000,1985.0,-37.849850,144.995580,4019.0
75%,4.000000,1.592425e+06,11.00000,3000.0,4.000000,3.000000,1.0,749.000000,337.000000,1985.0,-37.773575,145.099343,4019.0
max,5.000000,2.124300e+06,14.00000,3000.0,5.000000,3.000000,1.0,947.000000,426.200000,1985.0,-37.700650,145.199820,4019.0


### Етап 3: Перша модель машинного навчання (Decision Tree Regressor)

In [2]:
from sklearn.tree import DecisionTreeRegressor

# Вибір цільової змінної та ознак
y = melbourne_data.Price
melbourne_features = ['Rooms', 'Bathroom', 'Landsize', 'Lattitude', 'Longtitude']
X = melbourne_data[melbourne_features]

# Створення та навчання моделі
melbourne_model = DecisionTreeRegressor(random_state=1)
melbourne_model.fit(X, y)

print("Ознаки перших 5 будинків:")
print(X.head())
print("\nПрогноз вартості моделлю:")
print(melbourne_model.predict(X.head()))

Ознаки перших 5 будинків:
   Rooms  Bathroom  Landsize  Lattitude  Longtitude
0      4         1       264  -37.90094   145.07041
1      5         1       657  -37.77297   145.08265
2      3         3       767  -37.84418   145.04400
3      5         2       422  -37.93854   144.92516
4      5         1       863  -37.73665   144.90844

Прогноз вартості моделлю:
[1252000. 1710200. 1509000. 1803700. 1815100.]


### Етап 4: Валідація моделі (Model Validation & MAE)

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

train_X, val_X, train_y, val_y = train_test_split(X, y, random_state=0)

val_model = DecisionTreeRegressor(random_state=1)
val_model.fit(train_X, train_y)

val_predictions = val_model.predict(val_X)
mae_tree = mean_absolute_error(val_y, val_predictions)
print(f"Середня абсолютна похибка (MAE) дерева рішень: ${mae_tree:,.2f}")

Середня абсолютна похибка (MAE) дерева рішень: $49,874.80


### Етап 5: Дослідження Underfitting та Overfitting

In [4]:
def get_mae(max_leaf_nodes, train_X, val_X, train_y, val_y):
    model = DecisionTreeRegressor(max_leaf_nodes=max_leaf_nodes, random_state=0)
    model.fit(train_X, train_y)
    preds_val = model.predict(val_X)
    return mean_absolute_error(val_y, preds_val)

for max_leaf_nodes in [5, 50, 500, 5000]:
    my_mae = get_mae(max_leaf_nodes, train_X, val_X, train_y, val_y)
    print(f"Max leaf nodes: {max_leaf_nodes:<5} | MAE: ${my_mae:,.2f}")

Max leaf nodes: 5     | MAE: $127,633.38
Max leaf nodes: 50    | MAE: $42,897.82
Max leaf nodes: 500   | MAE: $51,269.98
Max leaf nodes: 5000  | MAE: $51,638.40


### Етап 6: Використання Random Forest Regressor

In [5]:
from sklearn.ensemble import RandomForestRegressor

forest_model = RandomForestRegressor(random_state=1)
forest_model.fit(train_X, train_y)
melb_preds = forest_model.predict(val_X)

mae_forest = mean_absolute_error(val_y, melb_preds)
print(f"Середня абсолютна похибка (MAE) Random Forest: ${mae_forest:,.2f}")

Середня абсолютна похибка (MAE) Random Forest: $38,933.08


### Етап 7: Фінальний прогноз вартості житла

In [6]:
sample_houses = val_X.head(5)
real_prices = val_y.head(5).values
predicted_prices = forest_model.predict(sample_houses)

print("=== Порівняння прогнозованої вартості з реальною ===\n")
for i in range(len(predicted_prices)):
    diff = predicted_prices[i] - real_prices[i]
    print(f"Будинок {i+1}: Прогноз = ${predicted_prices[i]:,.2f} | Реальна = ${real_prices[i]:,.2f} | Відхилення = ${diff:,.2f}")

=== Порівняння прогнозованої вартості з реальною ===

Будинок 1: Прогноз = $1,085,967.00 | Реальна = $1,050,000.00 | Відхилення = $35,967.00
Будинок 2: Прогноз = $1,504,711.00 | Реальна = $1,554,300.00 | Відхилення = $-49,589.00
Будинок 3: Прогноз = $1,547,716.00 | Реальна = $1,524,500.00 | Відхилення = $23,216.00
Будинок 4: Прогноз = $2,041,524.00 | Реальна = $2,029,100.00 | Відхилення = $12,424.00
Будинок 5: Прогноз = $1,695,323.00 | Реальна = $1,741,500.00 | Відхилення = $-46,177.00
